<a target="_blank" href="https://colab.research.google.com/github/cns-iu/hra-vccf-cell-distance-visualizations/blob/main/CDE_Demo_CIFAR_Gary_Bader.ipynb">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>

In [ ]:
#Install and import external packages
%pip install matplotlib pandas ipywidgets hra_jupyter_widgets

In [ ]:
# Import native packages
import os
from pprint import pprint
import random

In [ ]:
import pandas as pd
import ipywidgets as widgets

# Import hra-jupyter-widgets. For documentation, please see https://github.com/x-atlas-consortia/hra-jupyter-widgets/blob/main/usage.ipynb
from hra_jupyter_widgets import CdeVisualization

## Download data from Image Store.

Use curl to download the file contining all file urls, then download all files.

In [ ]:
!curl -L https://cdn.humanatlas.io/image-store/cifar-data/gary-bader-data/__s3_files-cifar-bader.csv -o __s3_files-cifar-bader.csv

# Make sure the data folder is present
folder_path = "data"

if not os.path.exists(folder_path):
    os.makedirs(folder_path)
    print(f"Folder '{folder_path}' created.")
else:
    print(f"Folder '{folder_path}' already exists.")

# Read csv as dataframe.
df = pd.read_csv('/content/__s3_files-cifar-bader.csv', header=None)

# Iterate through df.
for i in range(len(df)):
  fileurl = df.iloc[i, 0]
  print(f"FILEURL: {fileurl}")
  filename = fileurl.split('/')[-1]
  # Define the path to the file.
  file_path = f'{folder_path}/{filename}'
  # Check if the file exists
  if not os.path.exists(file_path):
      # If the file doesn't exist, run the curl command
      !curl -L {fileurl} -o {file_path}
      print(f"File downloaded and saved at {file_path}")
  else:
      print(f"File already exists at {file_path}")

## Read data as DataFrame

In [ ]:
len(os.listdir("data/"))

# Read metadata

In [ ]:
# Read the CSV file and convert it to a df
metadata = pd.read_csv(f"data/Sample_QC_spatial_Sample_QC.csv")
metadata.head(10)

# Read nodes file for given individual. Change `Individual_ID` value based on `Individual` in the table above.

In [ ]:
Individual_ID = "C105" # Change only this value based on values in the `Individual` column in the table above.

if Individual_ID == "C105" or Individual_ID == "C85":
  node_filepath = f"data/Pediatric_{Individual_ID}_cde.csv"
  print(node_filepath)
else:
  node_filepath = f"data/Adult_{Individual_ID}_cde.csv"
  print(node_filepath)

In [ ]:
# Read the CSV file and convert it to a df
df_nodes = pd.read_csv(node_filepath, header=0)
df_nodes.head()

# Display

In [ ]:
# Next, let's define a function that turns a DataFrame into a node list that can then be passed into the CdeVisualization or NodeDistVis widget
def make_node_list(df:pd.DataFrame, is_3d:bool = False):
  """Turn a DataFrame into a list of dicts for passing them into a HRA widget

  Args:
      df (pd.DataFrame): A DataFrame with cells
  """

  # If the df does not have a z-axis column, let's add one and set all cells to 0
  if not is_3d:
    df.loc[:, ('z')] = 0

  node_list = [{'x': row['x'], 'y': row['y'], 'z': row['z'], 'Cell Type': row['Cell Type']}
                 for index, row in df.iterrows()]

  return node_list

In [ ]:
# Read a random file from the output data path.
node_list = make_node_list(df_nodes, False)
# edge_list = [["Cell ID","Target ID", "X1", "Y1", "Z1", "X2", "Y2", "Z2"]] # this makes an empty edges list

In [ ]:
# Check if df_nodes has "ECs" or "Endothelial cells" in Cell Type column. Assign whichever exists to anchor_cell_type variable.
if "VEC" in df_nodes['Cell Type'].values:
    anchor_cell_type = "VEC"
elif "Endothelial cells" in df_nodes['Cell Type'].values:
    anchor_cell_type = "Endothelial cells"
else:
    anchor_cell_type = None

In [ ]:
# Finally, let's instantiate the CDEVisualization class with our node_list as parameter.
cde = CdeVisualization(
    node_target_selector=anchor_cell_type,
    max_edge_distance=200,
    nodes=node_list,
    node_target_key = "Cell Type",
)

# Display our new widget
display(cde)